# 02. Modelo predictivo

**Responsable:** Persona 3.  
**Entradas:** `data/processed/clean_model_dataset.csv` y `data/processed/team_features_2026.csv`, generados por `01_limpieza_datos.ipynb`.  
**Salidas:** probabilidades por seleccion, simulacion de torneo y prediccion de podio para Power BI.

El objetivo de esta seccion es entrenar un modelo supervisado con partidos historicos, estimar la fuerza relativa de las selecciones disponibles para 2026 y convertir esas probabilidades en un escenario de torneo reproducible mediante simulacion Monte Carlo.

## 1. Enfoque metodologico

Se modela cada partido como un problema multiclase con tres resultados posibles: victoria del local/equipo A, empate o victoria del visitante/equipo B. Para evitar fuga de informacion, las variables de forma y Elo ya fueron construidas por Persona 2 usando solo informacion disponible antes de cada partido historico.

La evaluacion se hace con separacion cronologica: los partidos mas antiguos entrenan y los mas recientes validan. La metrica principal es `log_loss`, porque penaliza probabilidades mal calibradas; `accuracy` se reporta como apoyo, pero no es suficiente para una prediccion probabilistica.

In [1]:
from pathlib import Path
from functools import lru_cache
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
N_SIMULATIONS = 5000
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../outputs_powerbi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MATCH_FILE = DATA_DIR / "clean_model_dataset.csv"
TEAM_FILE = DATA_DIR / "team_features_2026.csv"
QUALIFIED_FILE = DATA_DIR / "world_cup_2026_teams.csv"

for file in [MATCH_FILE, TEAM_FILE]:
    if not file.exists():
        raise FileNotFoundError(
            f"No se encontro {file}. Ejecuta primero notebooks/01_limpieza_datos.ipynb "
            "o descarga data/processed desde Drive."
        )

In [2]:
matches = pd.read_csv(MATCH_FILE, parse_dates=["date"])
teams_2026 = pd.read_csv(TEAM_FILE, parse_dates=["elo_date", "last_match_date"])

print("Partidos limpios:", matches.shape)
print("Selecciones 2026:", teams_2026.shape)
print("Rango temporal:", matches["date"].min().date(), "a", matches["date"].max().date())

display(matches.head())
display(teams_2026.sort_values("fifa_rank").head(10))

Partidos limpios: (49505, 24)
Selecciones 2026: (211, 10)
Rango temporal: 1872-11-30 a 2026-07-11


,match_id,date,home_team,away_team,home_score,away_score,goal_diff,result,neutral,tournament,tournament_tier,decided_by_penalties,home_form_points_avg,home_form_gf_avg,home_form_ga_avg,home_matches_played_before,away_form_points_avg,away_form_gf_avg,away_form_ga_avg,away_matches_played_before,home_elo_pre_match,away_elo_pre_match,elo_diff,has_min_history
0,0,1872-11-30,Scotland,England,0.0000,0.0000,0.0000,draw,0,Friendly,1,False,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,False
1,1,1873-03-08,England,Scotland,4.0000,2.0000,2.0000,home_win,0,Friendly,1,False,1.0000,0.0000,0.0000,1,1.0000,0.0000,0.0000,1,"2,003.0000","1,997.0000",6.0000,False
2,2,1874-03-07,Scotland,England,2.0000,1.0000,1.0000,home_win,0,Friendly,1,False,0.5000,1.0000,2.0000,2,2.0000,2.0000,1.0000,2,"1,986.0000","2,014.0000",-28.0000,False
3,3,1875-03-06,England,Scotland,2.0000,2.0000,0.0000,draw,0,Friendly,1,False,1.3333,1.6667,1.3333,3,1.3333,1.3333,1.6667,3,"2,006.0000","1,994.0000",12.0000,False
4,4,1876-03-04,Scotland,England,3.0000,0.0000,3.0000,home_win,0,Friendly,1,False,1.2500,1.5000,1.7500,4,1.2500,1.7500,1.5000,4,"1,997.0000","2,003.0000",-6.0000,False


,team,fifa_rank,fifa_points,confederation,elo_date,elo_rating,recent_points_avg,recent_gf_avg,recent_ga_avg,last_match_date
0,France,1,"1,925.8610",UEFA,2025-12-13,"2,062.0000",2.5000,2.5556,0.7778,2026-07-09
1,Argentina,2,"1,925.1490",CONMEBOL,2025-12-13,"2,113.0000",2.8000,2.8889,0.6667,2026-07-11
2,Spain,3,"1,912.3390",UEFA,2025-12-13,"2,171.0000",2.2000,1.7778,0.2222,2026-07-10
3,England,4,"1,871.3850",UEFA,2025-12-13,"2,042.0000",2.1000,1.7778,0.7778,2026-07-11
4,Brazil,5,"1,804.9230",CONMEBOL,2025-12-13,"1,979.0000",2.0000,2.3000,1.1000,2026-07-05
5,Morocco,6,"1,803.9880",CAF,2025-12-13,"1,830.0000",2.4000,2.4444,0.6667,2026-07-09
6,Portugal,7,"1,787.8550",UEFA,2025-12-13,"1,976.0000",2.1000,2.3000,0.6000,2026-07-06
7,Belgium,8,"1,778.3550",UEFA,2025-12-13,"1,849.0000",2.2000,2.8889,0.8889,2026-07-10
8,Netherlands,9,"1,775.5420",UEFA,2025-12-13,"1,959.0000",1.8000,2.1000,1.0000,2026-06-29
9,Mexico,10,"1,754.2970",CONCACAF,2025-12-13,"1,835.0000",2.3000,1.9000,0.5000,2026-07-05


## 2. Preparacion para entrenamiento

Se conservan solo partidos con historial minimo para ambos equipos y con variables numericas completas o imputables. No se usan goles finales como variables predictoras porque son parte del resultado que se quiere predecir.

In [3]:
FEATURES = [
    "neutral",
    "tournament_tier",
    "home_form_points_avg",
    "home_form_gf_avg",
    "home_form_ga_avg",
    "home_matches_played_before",
    "away_form_points_avg",
    "away_form_gf_avg",
    "away_form_ga_avg",
    "away_matches_played_before",
    "home_elo_pre_match",
    "away_elo_pre_match",
    "elo_diff",
]
TARGET = "result"
CLASS_ORDER = ["home_win", "draw", "away_win"]

train_data = matches.copy()
train_data = train_data[train_data["has_min_history"].astype(bool)]
train_data = train_data.dropna(subset=[TARGET])
train_data = train_data.sort_values("date").reset_index(drop=True)

missing_features = [c for c in FEATURES if c not in train_data.columns]
if missing_features:
    raise ValueError(f"Faltan columnas requeridas en el dataset limpio: {missing_features}")

split_idx = int(len(train_data) * 0.80)
X_train = train_data.loc[: split_idx - 1, FEATURES]
y_train = train_data.loc[: split_idx - 1, TARGET]
X_valid = train_data.loc[split_idx:, FEATURES]
y_valid = train_data.loc[split_idx:, TARGET]

print("Entrenamiento:", X_train.shape, "Validacion:", X_valid.shape)
print("Fecha corte validacion:", train_data.loc[split_idx, "date"].date())
print("Distribucion target entrenamiento:")
print(y_train.value_counts(normalize=True).reindex(CLASS_ORDER).round(3))

Entrenamiento: (38552, 13) Validacion: (9639, 13)
Fecha corte validacion: 2016-06-04
Distribucion target entrenamiento:
result
home_win   0.4970
draw       0.2170
away_win   0.2850
Name: proportion, dtype: float64


In [4]:
def numeric_pipeline(model):
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

models = {
    "regresion_logistica": numeric_pipeline(
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "random_forest": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "gradient_boosting": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.04,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        )),
    ]),
}

rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    valid_proba = model.predict_proba(X_valid)
    valid_pred = model.predict(X_valid)
    rows.append({
        "modelo": name,
        "log_loss": log_loss(y_valid, valid_proba, labels=model.classes_),
        "accuracy": accuracy_score(y_valid, valid_pred),
    })

model_results = pd.DataFrame(rows).sort_values("log_loss").reset_index(drop=True)
display(model_results)

best_model_name = model_results.loc[0, "modelo"]
best_model = fitted_models[best_model_name]
print("Modelo seleccionado:", best_model_name)
print(classification_report(y_valid, best_model.predict(X_valid), labels=CLASS_ORDER, zero_division=0))

,modelo,log_loss,accuracy
0,gradient_boosting,0.8746,0.6091
1,random_forest,0.8996,0.5842
2,regresion_logistica,0.9270,0.5530


Modelo seleccionado: gradient_boosting


              precision    recall  f1-score   support

    home_win       0.62      0.87      0.73      4685
        draw       0.27      0.01      0.01      2074
    away_win       0.59      0.61      0.60      2880

    accuracy                           0.61      9639
   macro avg       0.49      0.50      0.45      9639
weighted avg       0.54      0.61      0.54      9639



In [5]:
# Re-entrenamiento final con todos los partidos historicos disponibles para maximizar informacion.
final_model = models[best_model_name]
final_model.fit(train_data[FEATURES], train_data[TARGET])
MODEL_CLASSES = list(final_model.classes_)
print("Clases del modelo:", MODEL_CLASSES)

Clases del modelo: ['away_win', 'draw', 'home_win']


## 3. Selecciones consideradas para Mundial 2026

Si existe `data/processed/world_cup_2026_teams.csv`, se usa como lista definida de participantes. Debe contener una columna `team` y, si tambien incluye `group`, el simulador respeta esos grupos para la fase inicial. Si no existe, se construye un escenario reproducible con las 48 selecciones mejor posicionadas en ranking FIFA dentro de `team_features_2026.csv`.

In [6]:
teams_ranked = teams_2026.copy()
teams_ranked = teams_ranked.dropna(subset=["team", "fifa_rank"])
teams_ranked["fifa_rank"] = pd.to_numeric(teams_ranked["fifa_rank"], errors="coerce")
teams_ranked = teams_ranked.sort_values("fifa_rank").reset_index(drop=True)

if QUALIFIED_FILE.exists():
    qualified = pd.read_csv(QUALIFIED_FILE)
    if "team" not in qualified.columns:
        raise ValueError("world_cup_2026_teams.csv debe contener una columna llamada 'team'.")

    keep_cols = ["team"] + (["group"] if "group" in qualified.columns else [])
    tournament_teams = qualified[keep_cols].merge(teams_ranked, on="team", how="left")
    missing = tournament_teams.loc[tournament_teams["fifa_rank"].isna(), "team"].tolist()
    if missing:
        raise ValueError(f"Estas selecciones no existen en team_features_2026.csv: {missing}")

    if "group" in tournament_teams.columns:
        group_counts = tournament_teams.groupby("group")["team"].count().sort_index()
        if len(group_counts) != 12 or not (group_counts == 4).all():
            raise ValueError("world_cup_2026_teams.csv debe tener 12 grupos con 4 selecciones cada uno.")
        scenario_note = "Lista de clasificados y grupos leida desde world_cup_2026_teams.csv."
    else:
        scenario_note = "Lista de clasificados leida desde world_cup_2026_teams.csv; grupos generados por ranking FIFA."
else:
    tournament_teams = teams_ranked.head(48).copy()
    scenario_note = "Escenario proxy: 48 mejores selecciones por ranking FIFA disponible."

if tournament_teams["team"].duplicated().any():
    raise ValueError("Hay selecciones duplicadas en la lista de torneo.")

if len(tournament_teams) != 48:
    raise ValueError(f"El simulador espera 48 selecciones; se recibieron {len(tournament_teams)}.")

print(scenario_note)
display_cols = [c for c in ["group", "team", "fifa_rank", "fifa_points", "elo_rating", "recent_points_avg"] if c in tournament_teams.columns]
display(tournament_teams[display_cols].head(48))

Lista de clasificados y grupos leida desde world_cup_2026_teams.csv.


,group,team,fifa_rank,fifa_points,elo_rating,recent_points_avg
0,Grupo A,Mexico,10,"1,754.2970","1,835.0000",2.3000
1,Grupo A,South Africa,54,"1,451.2420","1,531.0000",1.2000
2,Grupo A,South Korea,32,"1,558.7190","1,784.0000",1.8000
3,Grupo A,Czech Republic,48,"1,467.2600","1,731.0000",1.9000
4,Grupo B,Canada,30,"1,571.3410","1,802.0000",1.6000
5,Grupo B,Bosnia and Herzegovina,61,"1,408.9300","1,571.0000",1.6000
6,Grupo B,Qatar,59,"1,411.0640","1,427.0000",0.6000
7,Grupo B,Switzerland,14,"1,710.8850","1,897.0000",1.9000
8,Grupo C,Brazil,5,"1,804.9230","1,979.0000",2.0000
9,Grupo C,Morocco,6,"1,803.9880","1,830.0000",2.4000


## 4. Probabilidad de partido entre dos selecciones

Para simular partidos neutrales de Mundial se construyen las mismas columnas usadas en entrenamiento. `team_a` se trata como el equivalente del local historico, aunque el partido se marca como neutral. Para reducir sesgos de orden, las fases de eliminacion alternan el orden de los equipos segun la llave.

In [7]:
team_lookup = teams_2026.set_index("team").to_dict(orient="index")
feature_medians = train_data[FEATURES].median(numeric_only=True)

TEAM_FEATURE_MAP = {
    "home_form_points_avg": "recent_points_avg",
    "home_form_gf_avg": "recent_gf_avg",
    "home_form_ga_avg": "recent_ga_avg",
    "home_elo_pre_match": "elo_rating",
    "away_form_points_avg": "recent_points_avg",
    "away_form_gf_avg": "recent_gf_avg",
    "away_form_ga_avg": "recent_ga_avg",
    "away_elo_pre_match": "elo_rating",
}

def _value(team, field, default=np.nan):
    return team_lookup.get(team, {}).get(field, default)

def build_match_row(team_a, team_b, tournament_tier=4, neutral=1):
    row = {
        "neutral": neutral,
        "tournament_tier": tournament_tier,
        "home_form_points_avg": _value(team_a, "recent_points_avg"),
        "home_form_gf_avg": _value(team_a, "recent_gf_avg"),
        "home_form_ga_avg": _value(team_a, "recent_ga_avg"),
        "home_matches_played_before": feature_medians["home_matches_played_before"],
        "away_form_points_avg": _value(team_b, "recent_points_avg"),
        "away_form_gf_avg": _value(team_b, "recent_gf_avg"),
        "away_form_ga_avg": _value(team_b, "recent_ga_avg"),
        "away_matches_played_before": feature_medians["away_matches_played_before"],
        "home_elo_pre_match": _value(team_a, "elo_rating"),
        "away_elo_pre_match": _value(team_b, "elo_rating"),
    }
    row["elo_diff"] = row["home_elo_pre_match"] - row["away_elo_pre_match"]
    return pd.DataFrame([row], columns=FEATURES)

@lru_cache(maxsize=None)
def match_probabilities(team_a, team_b):
    proba = final_model.predict_proba(build_match_row(team_a, team_b))[0]
    p = dict(zip(MODEL_CLASSES, proba))
    return {cls: float(p.get(cls, 0.0)) for cls in CLASS_ORDER}

def knockout_winner(team_a, team_b, rng):
    p = match_probabilities(team_a, team_b)
    # En eliminacion, un empate se resuelve por tiempo extra/penales. Se reparte segun fuerza relativa.
    decisive_a = p["home_win"] + 0.5 * p["draw"]
    decisive_b = p["away_win"] + 0.5 * p["draw"]
    prob_a = decisive_a / (decisive_a + decisive_b)
    return team_a if rng.random() < prob_a else team_b

def group_match(team_a, team_b, rng):
    p = match_probabilities(team_a, team_b)
    result = rng.choice(CLASS_ORDER, p=[p[c] for c in CLASS_ORDER])
    if result == "home_win":
        return team_a, team_b, 3, 0
    if result == "away_win":
        return team_a, team_b, 0, 3
    return team_a, team_b, 1, 1

# Ejemplo de lectura del modelo
example_a, example_b = tournament_teams.loc[0, "team"], tournament_teams.loc[1, "team"]
print(example_a, "vs", example_b, match_probabilities(example_a, example_b))

Mexico vs South Africa {'home_win': 0.6630982153514743, 'draw': 0.23168314470362025, 'away_win': 0.10521863994490545}


## 5. Simulacion Monte Carlo del torneo

La estructura usada es compatible con el Mundial 2026: 48 equipos, 12 grupos de 4, avanzan los dos primeros de cada grupo y los 8 mejores terceros; luego eliminacion directa desde ronda de 32. Como el calendario oficial no forma parte de los insumos entregados, los grupos se generan por bombos basados en ranking FIFA y distribucion serpentina.

In [8]:
def make_seeded_groups(seed_df):
    if "group" in seed_df.columns:
        grouped = seed_df.sort_values(["group", "fifa_rank"]).groupby("group", sort=True)["team"].apply(list).to_dict()
        return grouped

    ordered = seed_df.sort_values("fifa_rank")["team"].tolist()
    pots = [ordered[i:i + 12] for i in range(0, 48, 12)]
    groups = {f"Grupo {chr(65 + i)}": [] for i in range(12)}
    group_names = list(groups.keys())
    for pot_idx, pot in enumerate(pots):
        names = group_names if pot_idx % 2 == 0 else list(reversed(group_names))
        for group, team in zip(names, pot):
            groups[group].append(team)
    return groups

def simulate_group(group_name, group_teams, rng):
    table = {team: {"team": team, "group": group_name, "points": 0, "wins": 0} for team in group_teams}
    for i in range(len(group_teams)):
        for j in range(i + 1, len(group_teams)):
            a, b = group_teams[i], group_teams[j]
            _, _, pts_a, pts_b = group_match(a, b, rng)
            table[a]["points"] += pts_a
            table[b]["points"] += pts_b
            table[a]["wins"] += int(pts_a == 3)
            table[b]["wins"] += int(pts_b == 3)
    result = pd.DataFrame(table.values())
    # Desempate reproducible: puntos, victorias y ranking FIFA.
    ranks = tournament_teams.set_index("team")["fifa_rank"].to_dict()
    result["fifa_rank"] = result["team"].map(ranks)
    return result.sort_values(["points", "wins", "fifa_rank"], ascending=[False, False, True]).reset_index(drop=True)

def simulate_tournament(seed_df, rng):
    groups = make_seeded_groups(seed_df)
    group_tables = []
    for group_name, group_teams in groups.items():
        table = simulate_group(group_name, group_teams, rng)
        table["group_position"] = np.arange(1, len(table) + 1)
        group_tables.append(table)
    all_groups = pd.concat(group_tables, ignore_index=True)

    qualified = all_groups[all_groups["group_position"] <= 2].copy()
    thirds = all_groups[all_groups["group_position"] == 3].sort_values(
        ["points", "wins", "fifa_rank"], ascending=[False, False, True]
    ).head(8)
    qualified = pd.concat([qualified, thirds], ignore_index=True)
    qualified = qualified.sort_values(["group_position", "points", "wins", "fifa_rank"], ascending=[True, False, False, True])
    round32 = qualified["team"].tolist()[:32]

    def play_round(teams):
        winners, losers = [], []
        for i in range(0, len(teams), 2):
            a, b = teams[i], teams[i + 1]
            winner = knockout_winner(a, b, rng)
            loser = b if winner == a else a
            winners.append(winner)
            losers.append(loser)
        return winners, losers

    r16, _ = play_round(round32)
    qf, _ = play_round(r16)
    semifinalists, qf_losers = play_round(qf)
    finalists, semifinal_losers = play_round(semifinalists)
    champion_list, final_loser = play_round(finalists)
    champion = champion_list[0]
    runner_up = final_loser[0]
    third_place = knockout_winner(semifinal_losers[0], semifinal_losers[1], rng)
    fourth_place = semifinal_losers[1] if third_place == semifinal_losers[0] else semifinal_losers[0]

    return {
        "round32": round32,
        "round16": r16,
        "quarterfinalists": qf,
        "semifinalists": semifinalists,
        "finalists": finalists,
        "champion": champion,
        "runner_up": runner_up,
        "third_place": third_place,
        "fourth_place": fourth_place,
    }

rng = np.random.default_rng(RANDOM_STATE)
base_groups = make_seeded_groups(tournament_teams)
base_groups

{'Grupo A': ['Mexico', 'South Korea', 'Czech Republic', 'South Africa'],
 'Grupo B': ['Switzerland', 'Canada', 'Qatar', 'Bosnia and Herzegovina'],
 'Grupo C': ['Brazil', 'Morocco', 'Scotland', 'Haiti'],
 'Grupo D': ['United States', 'Turkey', 'Australia', 'Paraguay'],
 'Grupo E': ['Germany', 'Ecuador', 'Ivory Coast', 'Curaçao'],
 'Grupo F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
 'Grupo G': ['Belgium', 'Iran', 'Egypt', 'New Zealand'],
 'Grupo H': ['Spain', 'Uruguay', 'Saudi Arabia', 'Cape Verde'],
 'Grupo I': ['France', 'Senegal', 'Norway', 'Iraq'],
 'Grupo J': ['Argentina', 'Austria', 'Algeria', 'Jordan'],
 'Grupo K': ['Portugal', 'Colombia', 'DR Congo', 'Uzbekistan'],
 'Grupo L': ['England', 'Croatia', 'Panama', 'Ghana']}

In [9]:
summary = {team: {
    "team": team,
    "p_round32": 0,
    "p_round16": 0,
    "p_quarterfinal": 0,
    "p_semifinal": 0,
    "p_final": 0,
    "p_champion": 0,
    "p_runner_up": 0,
    "p_third_place": 0,
    "p_fourth_place": 0,
} for team in tournament_teams["team"]}

for _ in range(N_SIMULATIONS):
    sim = simulate_tournament(tournament_teams, rng)
    for team in sim["round32"]:
        summary[team]["p_round32"] += 1
    for team in sim["round16"]:
        summary[team]["p_round16"] += 1
    for team in sim["quarterfinalists"]:
        summary[team]["p_quarterfinal"] += 1
    for team in sim["semifinalists"]:
        summary[team]["p_semifinal"] += 1
    for team in sim["finalists"]:
        summary[team]["p_final"] += 1
    summary[sim["champion"]]["p_champion"] += 1
    summary[sim["runner_up"]]["p_runner_up"] += 1
    summary[sim["third_place"]]["p_third_place"] += 1
    summary[sim["fourth_place"]]["p_fourth_place"] += 1

probabilities = pd.DataFrame(summary.values())
prob_cols = [c for c in probabilities.columns if c.startswith("p_")]
probabilities[prob_cols] = probabilities[prob_cols] / N_SIMULATIONS
probabilities = probabilities.merge(
    tournament_teams[["team", "fifa_rank", "fifa_points", "elo_rating", "confederation"]],
    on="team",
    how="left",
)
probabilities = probabilities.sort_values("p_champion", ascending=False).reset_index(drop=True)

display(probabilities.head(20))

,team,p_round32,p_round16,p_quarterfinal,p_semifinal,p_final,p_champion,p_runner_up,p_third_place,p_fourth_place,fifa_rank,fifa_points,elo_rating,confederation
0,Argentina,0.9890,0.6154,0.4220,0.2854,0.1950,0.1400,0.0550,0.0714,0.0190,2,"1,925.1490","2,113.0000",CONMEBOL
1,Spain,0.9942,0.6254,0.4074,0.2598,0.1744,0.1220,0.0524,0.0650,0.0204,3,"1,912.3390","2,171.0000",UEFA
2,France,0.9814,0.6020,0.3794,0.2344,0.1454,0.0992,0.0462,0.0650,0.0240,1,"1,925.8610","2,062.0000",UEFA
3,Portugal,0.9590,0.5754,0.3440,0.1998,0.1192,0.0650,0.0542,0.0508,0.0298,7,"1,787.8550","1,976.0000",UEFA
4,England,0.9824,0.5394,0.3100,0.1768,0.0976,0.0616,0.0360,0.0506,0.0286,4,"1,871.3850","2,042.0000",UEFA
5,Colombia,0.9416,0.5600,0.3276,0.1878,0.1072,0.0592,0.0480,0.0532,0.0274,11,"1,739.8950","1,998.0000",CONMEBOL
6,Germany,0.9688,0.5432,0.2960,0.1558,0.0790,0.0420,0.0370,0.0462,0.0306,12,"1,726.2190","1,910.0000",UEFA
7,Brazil,0.9504,0.5212,0.2934,0.1594,0.0748,0.0394,0.0354,0.0546,0.0300,5,"1,804.9230","1,979.0000",CONMEBOL
8,Netherlands,0.9428,0.5026,0.2586,0.1296,0.0674,0.0318,0.0356,0.0348,0.0274,9,"1,775.5420","1,959.0000",UEFA
9,Morocco,0.9194,0.4710,0.2502,0.1284,0.0680,0.0282,0.0398,0.0320,0.0284,6,"1,803.9880","1,830.0000",CAF


## 6. Prediccion de podio

El podio se toma como la seleccion con mayor probabilidad marginal en cada posicion, evitando repetir equipo. Este criterio es estable para Power BI y facilita explicar la salida: no es una unica simulacion aislada, sino el resumen de miles de escenarios.

In [10]:
def pick_unique(df, prob_col, used):
    for _, row in df.sort_values(prob_col, ascending=False).iterrows():
        if row["team"] not in used:
            used.add(row["team"])
            return row
    raise RuntimeError("No fue posible construir podio sin duplicados.")

used = set()
podium_rows = [
    ("Campeon", pick_unique(probabilities, "p_champion", used), "p_champion"),
    ("Subcampeon", pick_unique(probabilities, "p_runner_up", used), "p_runner_up"),
    ("Tercer lugar", pick_unique(probabilities, "p_third_place", used), "p_third_place"),
    ("Cuarto lugar", pick_unique(probabilities, "p_fourth_place", used), "p_fourth_place"),
]

podium = pd.DataFrame([
    {
        "position": i + 1,
        "label": label,
        "team": row["team"],
        "probability": row[prob_col],
        "fifa_rank": row["fifa_rank"],
        "elo_rating": row["elo_rating"],
    }
    for i, (label, row, prob_col) in enumerate(podium_rows)
])

display(podium)

,position,label,team,probability,fifa_rank,elo_rating
0,1,Campeon,Argentina,0.1400,2,"2,113.0000"
1,2,Subcampeon,Portugal,0.0542,7,"1,976.0000"
2,3,Tercer lugar,Spain,0.0650,3,"2,171.0000"
3,4,Cuarto lugar,Norway,0.0414,19,"1,922.0000"


## 7. Exportacion de resultados

Se guardan archivos livianos para la Persona 4 y Power BI:

- `outputs_powerbi/team_probabilities.csv`: probabilidades por seleccion.
- `outputs_powerbi/predicted_podium.csv`: podio proyectado.
- `outputs_powerbi/model_metrics.csv`: comparacion de modelos.
- `outputs_powerbi/model_metadata.json`: notas de escenario y configuracion.

In [11]:
probabilities.to_csv(OUTPUT_DIR / "team_probabilities.csv", index=False)
podium.to_csv(OUTPUT_DIR / "predicted_podium.csv", index=False)
model_results.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

metadata = {
    "responsable": "Persona 3",
    "modelo_seleccionado": best_model_name,
    "n_partidos_entrenamiento_final": int(len(train_data)),
    "n_simulaciones": N_SIMULATIONS,
    "escenario_equipos": scenario_note,
    "archivos_entrada": [str(MATCH_FILE), str(TEAM_FILE)],
    "archivos_salida": [
        str(OUTPUT_DIR / "team_probabilities.csv"),
        str(OUTPUT_DIR / "predicted_podium.csv"),
        str(OUTPUT_DIR / "model_metrics.csv"),
    ],
}
with open(OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Archivos exportados en", OUTPUT_DIR.resolve())
print(json.dumps(metadata, ensure_ascii=False, indent=2))

Archivos exportados en C:\Users\rodri\OneDrive\Documentos\Maestria\6_Sexto_trimestre\Mineria Avanzada\Proyecto_Mineria_Avanzada\outputs_powerbi
{
  "responsable": "Persona 3",
  "modelo_seleccionado": "gradient_boosting",
  "n_partidos_entrenamiento_final": 48191,
  "n_simulaciones": 5000,
  "escenario_equipos": "Lista de clasificados y grupos leida desde world_cup_2026_teams.csv.",
  "archivos_entrada": [
    "..\\data\\processed\\clean_model_dataset.csv",
    "..\\data\\processed\\team_features_2026.csv"
  ],
  "archivos_salida": [
    "..\\outputs_powerbi\\team_probabilities.csv",
    "..\\outputs_powerbi\\predicted_podium.csv",
    "..\\outputs_powerbi\\model_metrics.csv"
  ]
}


## 8. Conclusiones de Persona 3

El modelo transforma el historial de partidos en probabilidades interpretables y usa esas probabilidades para proyectar el avance de cada seleccion en un formato consumible por Power BI. La principal limitacion es que, si no se proporciona una lista/calendario oficial de clasificados, el torneo se simula con un escenario proxy basado en ranking FIFA. Al agregar `world_cup_2026_teams.csv`, el mismo notebook reutiliza automaticamente la lista oficial.